## 1. Setup e imports

In [ ]:
import re
import sys
import pandas as pd
from dotenv import load_dotenv

# Raiz del proyecto (ejecutar desde notebooks/)
sys.path.append("../")
load_dotenv("../.env")
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_columns", None)

from src.config.settings import COUNTRY
from src.config.brand_configs import BRAND_FEES
from src.core.price_tracking.price_tracking import run_price_tracking
from src.utils.clean_model_name import clean_model_name
from src.utils.replace_model_name import map_model_name, load_mapping_file
from src.core.italika_pipeline import build_price_comparison
from src.sources.sheets.reader import GoogleSheetReader
from src.core.scraper.app import ScrapingUtils

gsheets = GoogleSheetReader()
scraper = ScrapingUtils()

In [2]:
urls_scraped = scraper.get_all_urls_from_website("https://www.motosbajaj.com.mx/")

In [3]:
urls = []
for url in urls_scraped:
    if "product-page" in url:
        urls.append(url)

## 2. Configuracion

In [ ]:
BRAND_NAME = "Italika"

# URLs definidas manualmente por ejecucion.
urls_to_scrape = urls

## Lectura de base de inventario

Lectura desde archivo local

In [5]:
# CSV_PATH = rf"C:\Users\JTRUJILLO\Desktop\utiles\Reportes\historical_data\src\data\prices\{COUNTRY}-prices.csv"
# df_inventory = pd.read_csv(CSV_PATH)

Lectura desde archivo en google sheets

In [6]:
google_sheet_info = {
    "sheet_name": '[MKP] Precios',
    "worksheet": f'price_data_mx',
}
df_inventory = gsheets.read_sheet(google_sheet_info)

## 3. Cargar inventario

In [7]:
df_inventory = df_inventory[df_inventory["brand"] == BRAND_NAME]
df_inventory = df_inventory[df_inventory["status"].isin(["available", "no_stock"])] # Modelos disponibles en marketplace

## 4. Definir URLs a scrapear (input manual)

In [8]:
if not urls_to_scrape:
    raise ValueError("Debes cargar al menos una URL en 'urls_to_scrape'")

print(f"URLs recibidas para scraping: {len(urls_to_scrape)}")
urls_to_scrape[:3]

URLs recibidas para scraping: 31


['https://www.motosbajaj.com.mx/product-page/pulsar-ls-125',
 'https://www.motosbajaj.com.mx/product-page/re-60-qute',
 'https://www.motosbajaj.com.mx/product-page/pulsar-n-250-ug']

In [9]:
urls_to_scrape = ['https://www.motosbajaj.com.mx/product-page/pulsar-ls-125',]
#  'https://www.motosbajaj.com.mx/product-page/n-160-carburada-2026',
#  'https://www.motosbajaj.com.mx/product-page/copia-de-pulsar-n-160-fi-abs-ug',
#  'https://www.motosbajaj.com.mx/product-page/boxer-bm-150',
#  'https://www.motosbajaj.com.mx/product-page/dominar-400',
#  'https://www.motosbajaj.com.mx/product-page/pulsar-ns-160-ug',
#  'https://www.motosbajaj.com.mx/product-page/pulsar-ns-400-z',
#  'https://www.motosbajaj.com.mx/product-page/pulsar-n-125-fi-cbs',
#  'https://www.motosbajaj.com.mx/product-page/dominar-400-touring-2026',
#  'https://www.motosbajaj.com.mx/product-page/pulsar-ns-200-ug-2026',
#  'https://www.motosbajaj.com.mx/product-page/pulsar-125-ns',
#  'https://www.motosbajaj.com.mx/product-page/pulsar-n-250-ug',
#  'https://www.motosbajaj.com.mx/product-page/maxima-z-fl-pasajeros',
#  'https://www.motosbajaj.com.mx/product-page/re4s-pasajeros',
#  'https://www.motosbajaj.com.mx/product-page/pulsar-200-rs',
#  'https://www.motosbajaj.com.mx/product-page/pulsar-n-250-carburada-2026',
#  'https://www.motosbajaj.com.mx/product-page/pulsar-n-160-fi-abs-ug-2026',
#  'https://www.motosbajaj.com.mx/product-page/pulsar-n-250-fi-abs-ug-2026',
#  'https://www.motosbajaj.com.mx/product-page/pulsar-ns-200-ug',
#  'https://www.motosbajaj.com.mx/product-page/avenger-220',
#  'https://www.motosbajaj.com.mx/product-page/pulsar-f-250',
#  'https://www.motosbajaj.com.mx/product-page/dominar-250',
#  'https://www.motosbajaj.com.mx/product-page/pulsar-n-160-fi-abs-premium',
# ]

## 5. Ejecutar price tracking

In [10]:
rows = run_price_tracking(
    urls=urls_to_scrape,
    brand_name=BRAND_NAME,
)

In [13]:
rows

[{'brand_name': 'Bajaj',
  'model_name': 'Pulsar LS 125',
  'year_scraped': 2025,
  'url': 'https://www.motosbajaj.com.mx/product-page/pulsar-ls-125',
  'price': '29,999.00',
  'price_type': 'oferta',
  'currency': 'MXN',
  'captured_at': '2026-04-08T15:01:30.610445+00:00',
  'change_status': '',
  'previous_scrape_at': '',
  'visibility': ''}]

In [14]:
df_scraped = pd.DataFrame(rows)

In [15]:
df_scraped

,brand_name,model_name,year_scraped,url,price,price_type,currency,captured_at,change_status,previous_scrape_at,visibility
0,Bajaj,Pulsar LS 125,2025,https://www.motosbajaj.com.mx/product-page/pulsar-ls-125,"29,999.00",oferta,MXN,2026-04-08T15:01:30.610445+00:00,,,


## 6. Limpiar y mapear modelos

In [ ]:
df_scraped["year_scraped"] = pd.to_numeric(df_scraped.get("year_scraped"), errors="coerce").astype("Int64")

df_scraped["model_name_clean"] = df_scraped["model_name"].apply(
    lambda model_name: clean_model_name(model_name, brand_name=BRAND_NAME)
)

mapeo = load_mapping_file(COUNTRY, BRAND_NAME)
df_scraped["model_mapped"] = df_scraped["model_name_clean"].apply(lambda x: map_model_name(x, mapeo))
df_scraped["model_mapped"] = df_scraped["model_mapped"].str.lower()

## 7. Merge con inventario (inspeccion)

In [13]:
# 1) Preparar inventario (marketplace)
inv_for_merge = df_inventory.copy()
inv_for_merge["model_lower"] = inv_for_merge["model"].str.lower()
inv_for_merge["year"] = pd.to_numeric(inv_for_merge["year"], errors="coerce")

# 2) Preparar scrapeado
scraped_for_merge = df_scraped.copy()
scraped_for_merge["year_scraped"] = pd.to_numeric(
    scraped_for_merge.get("year_scraped"),
    errors="coerce",
)

# 3) Merge por modelo + año
# Comentario en español: Outer conserva tanto marketplace sin match como scrapeado sin match.
df_merged = pd.merge(
    inv_for_merge,
    scraped_for_merge,
    left_on=["model_lower", "year"],
    right_on=["model_mapped", "year_scraped"],
    how="outer",
    indicator=True,
)

df_merged.head(2)

,date,code,brand,model,year,status,price_base,discount_amount,price_net,model_lower,brand_name,model_name,year_scraped,url,price,price_type,currency,captured_at,change_status,previous_scrape_at,visibility,model_name_clean,model_mapped,_merge
0,07/04/2026,MX1335-bajaj-avenger-220-cruise,Bajaj,Avenger 220 Cruise,2025.0,available,54499.0,3000.0,51499.0,avenger 220 cruise,Bajaj,Avenger 220 Cruise,2025,https://www.motosbajaj.com.mx/product-page/avenger-220,"51,499.00",oferta,MXN,2026-04-07T20:25:30.844251+00:00,same,2026-04-07T20:14:07.657+00:00,visible,Avenger 220 Cruise,avenger 220 cruise,both
1,07/04/2026,MX250-bajaj-boxer-bm-150,Bajaj,Boxer BM 150,2024.0,no_stock,30999.0,0.0,30999.0,boxer bm 150,NaN,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only


In [14]:
df_scraped[df_scraped["model_name"] == "Boxer BM 150"]

,brand_name,model_name,year_scraped,url,price,price_type,currency,captured_at,change_status,previous_scrape_at,visibility,model_name_clean,model_mapped
3,Bajaj,Boxer BM 150,2026,https://www.motosbajaj.com.mx/product-page/boxer-bm-150,"31,499.00",oferta,MXN,2026-04-07T20:25:30.844251+00:00,same,2026-04-07T20:13:45.405+00:00,visible,Boxer BM 150,boxer bm 150


In [15]:
df_inventory[df_inventory["model"] == "Boxer BM 150"]

,date,code,brand,model,year,status,price_base,discount_amount,price_net
802,07/04/2026,MX250-bajaj-boxer-bm-150,Bajaj,Boxer BM 150,2024,no_stock,30999.0,0.0,30999.0
1449,07/04/2026,MX250-bajaj-boxer-bm-150,Bajaj,Boxer BM 150,2025,no_stock,34499.0,3000.0,31499.0


In [16]:
df_scraped[df_scraped["model_name"] == "Boxer BM 150"]

,brand_name,model_name,year_scraped,url,price,price_type,currency,captured_at,change_status,previous_scrape_at,visibility,model_name_clean,model_mapped
3,Bajaj,Boxer BM 150,2026,https://www.motosbajaj.com.mx/product-page/boxer-bm-150,"31,499.00",oferta,MXN,2026-04-07T20:25:30.844251+00:00,same,2026-04-07T20:13:45.405+00:00,visible,Boxer BM 150,boxer bm 150


## 8. Construir comparacion de precios

In [ ]:
df_final = build_price_comparison(
    df_scraped,
    df_inventory,
    COUNTRY,
    galgo_fee=BRAND_FEES.get(BRAND_NAME.lower(), 0),
)

In [ ]:
df_final[df_final["url_scraped"].notna()]

,captured_at,previous_scrape_at,code,brand,model,model_name,year,year_scraped,status,price_scraped,price_scraped_with_galgo_fee,price_base,discount_amount,price_net,price_diff,price_type,currency,change_status,visibility,url_scraped,marketplace_url
0,2026-04-07T20:25:30.844251+00:00,2026-04-07T20:14:07.657+00:00,MX1335-bajaj-avenger-220-cruise,Bajaj,Avenger 220 Cruise,Avenger 220 Cruise,2025.0,2025,available,"51,499.00",51499.0,54499.0,3000.0,51499.0,0.0,oferta,MXN,same,visible,https://www.motosbajaj.com.mx/product-page/avenger-220,https://www.galgo.com/mx/motos/MX1335-bajaj-avenger-220-cruise
3,2026-04-07T20:25:30.844251+00:00,2026-04-07T20:13:45.405+00:00,NaN,NaN,NaN,Boxer BM 150,NaN,2026,NaN,"31,499.00",31499.0,NaN,NaN,NaN,NaN,oferta,MXN,same,visible,https://www.motosbajaj.com.mx/product-page/boxer-bm-150,https://www.galgo.com/mx/motos/nan
4,2026-04-07T20:25:30.844251+00:00,2026-04-07T20:14:14.43+00:00,MX246-bajaj-dominar-250-ug,Bajaj,Dominar 250,Dominar 250 2026,2026.0,2026,available,"73,499.00",73499.0,73499.0,0.0,73499.0,0.0,oferta,MXN,same,visible,https://www.motosbajaj.com.mx/product-page/dominar-250,https://www.galgo.com/mx/motos/MX246-bajaj-dominar-250-ug
6,2026-04-07T20:25:30.844251+00:00,2026-04-07T20:13:45.333+00:00,MX253-bajaj-dominar-400-te,Bajaj,Dominar 400 TE,Dominar 400 Touring,2025.0,2025,no_stock,"92,999.00",92999.0,95999.0,3000.0,92999.0,0.0,oferta,MXN,same,visible,https://www.motosbajaj.com.mx/product-page/dominar-400,https://www.galgo.com/mx/motos/MX253-bajaj-dominar-400-te
7,2026-04-07T20:25:30.844251+00:00,2026-04-07T20:13:52.6+00:00,MX253-bajaj-dominar-400-te,Bajaj,Dominar 400 TE,Dominar 400 Touring 2026,2026.0,2026,no_stock,"95,999.00",95999.0,95999.0,0.0,95999.0,0.0,oferta,MXN,same,visible,https://www.motosbajaj.com.mx/product-page/dominar-400-touring-2026,https://www.galgo.com/mx/motos/MX253-bajaj-dominar-400-te
9,2026-04-07T20:25:30.844251+00:00,2026-04-07T20:13:59.777+00:00,NaN,NaN,NaN,Maxima Z FL Pasajeros,NaN,2026,NaN,"116,500.00",116500.0,NaN,NaN,NaN,NaN,oferta,MXN,same,visible,https://www.motosbajaj.com.mx/product-page/maxima-z-fl-pasajeros,https://www.galgo.com/mx/motos/nan
10,2026-04-07T20:25:30.844251+00:00,2026-04-07T20:14:13.944+00:00,MX3265-bajaj-pulsar-f250,Bajaj,Pulsar F250,Pulsar F 250,2026.0,2026,available,"69,999.00",69999.0,71999.0,2000.0,69999.0,0.0,oferta,MXN,same,visible,https://www.motosbajaj.com.mx/product-page/pulsar-f-250,https://www.galgo.com/mx/motos/MX3265-bajaj-pulsar-f250
11,2026-04-07T20:25:30.844251+00:00,2026-04-07T20:13:45.451+00:00,NaN,NaN,NaN,Pulsar LS 125,NaN,2025,NaN,"29,999.00",29999.0,NaN,NaN,NaN,NaN,oferta,MXN,same,visible,https://www.motosbajaj.com.mx/product-page/pulsar-ls-125,https://www.galgo.com/mx/motos/nan
12,2026-04-07T20:25:30.844251+00:00,2026-04-07T20:13:52.722+00:00,MX3098-bajaj-pulsar-n-125,Bajaj,Pulsar N 125 FI CBS,Pulsar N 125 FI CBS,2026.0,2026,no_stock,"36,499.00",36499.0,37999.0,2500.0,35499.0,1000.0,oferta,MXN,same,visible,https://www.motosbajaj.com.mx/product-page/pulsar-n-125-fi-cbs,https://www.galgo.com/mx/motos/MX3098-bajaj-pulsar-n-125
13,2026-04-07T20:25:30.844251+00:00,2026-04-07T20:13:45.52+00:00,MX451-bajaj-pulsar-n-160,Bajaj,Pulsar N 160,Pulsar N 160 Carburada 2026,2026.0,2026,no_stock,"38,599.00",38599.0,40599.0,2000.0,38599.0,0.0,oferta,MXN,same,visible,https://www.motosbajaj.com.mx/product-page/n-160-carburada-2026,https://www.galgo.com/mx/motos/MX451-bajaj-pulsar-n-160


In [19]:
df_final[df_final["url_scraped"].isna()]

,captured_at,previous_scrape_at,code,brand,model,model_name,year,year_scraped,status,price_scraped,price_scraped_with_galgo_fee,price_base,discount_amount,price_net,price_diff,price_type,currency,change_status,visibility,url_scraped,marketplace_url
1,NaN,NaN,MX250-bajaj-boxer-bm-150,Bajaj,Boxer BM 150,NaN,2024.0,<NA>,no_stock,NaN,NaN,30999.0,0.0,30999.0,NaN,NaN,NaN,NaN,NaN,NaN,https://www.galgo.com/mx/motos/MX250-bajaj-boxer-bm-150
2,NaN,NaN,MX250-bajaj-boxer-bm-150,Bajaj,Boxer BM 150,NaN,2025.0,<NA>,no_stock,NaN,NaN,34499.0,3000.0,31499.0,NaN,NaN,NaN,NaN,NaN,NaN,https://www.galgo.com/mx/motos/MX250-bajaj-boxer-bm-150
5,NaN,NaN,MX3035-bajaj-dominar-400-4e,Bajaj,Dominar 400 4E,NaN,2025.0,<NA>,available,NaN,NaN,97999.0,0.0,97999.0,NaN,NaN,NaN,NaN,NaN,NaN,https://www.galgo.com/mx/motos/MX3035-bajaj-dominar-400-4e
8,NaN,NaN,MX2784-bajaj-maxima-z-fl-pasajeros,Bajaj,Maxima Z FL Pasajeros,NaN,2025.0,<NA>,no_stock,NaN,NaN,114000.0,0.0,114000.0,NaN,NaN,NaN,NaN,NaN,NaN,https://www.galgo.com/mx/motos/MX2784-bajaj-maxima-z-fl-pasajeros
16,NaN,NaN,MX2450-bajaj-n-250-fi-abs,Bajaj,Pulsar N 250 FI ABS,NaN,2026.0,<NA>,available,NaN,NaN,69999.0,4500.0,65499.0,NaN,NaN,NaN,NaN,NaN,NaN,https://www.galgo.com/mx/motos/MX2450-bajaj-n-250-fi-abs
21,NaN,NaN,MX1512-bajaj-pulsar-ns-125-ug,Bajaj,Pulsar NS 125 UG,NaN,2024.0,<NA>,no_stock,NaN,NaN,38999.0,0.0,38999.0,NaN,NaN,NaN,NaN,NaN,NaN,https://www.galgo.com/mx/motos/MX1512-bajaj-pulsar-ns-125-ug
24,NaN,NaN,MX1509-bajaj-pulsar-ns-160-ug,Bajaj,Pulsar NS 160 UG,NaN,2024.0,<NA>,no_stock,NaN,NaN,47199.0,0.0,47199.0,NaN,NaN,NaN,NaN,NaN,NaN,https://www.galgo.com/mx/motos/MX1509-bajaj-pulsar-ns-160-ug
25,NaN,NaN,MX1509-bajaj-pulsar-ns-160-ug,Bajaj,Pulsar NS 160 UG,NaN,2025.0,<NA>,available,NaN,NaN,49499.0,3000.0,46499.0,NaN,NaN,NaN,NaN,NaN,NaN,https://www.galgo.com/mx/motos/MX1509-bajaj-pulsar-ns-160-ug
30,NaN,NaN,MX2783-bajaj-re4s-pasajeros,Bajaj,RE4S Pasajeros,NaN,2025.0,<NA>,available,NaN,NaN,96500.0,0.0,96500.0,NaN,NaN,NaN,NaN,NaN,NaN,https://www.galgo.com/mx/motos/MX2783-bajaj-re4s-pasajeros


In [20]:
df_final

,captured_at,previous_scrape_at,code,brand,model,model_name,year,year_scraped,status,price_scraped,price_scraped_with_galgo_fee,price_base,discount_amount,price_net,price_diff,price_type,currency,change_status,visibility,url_scraped,marketplace_url
0,2026-04-07T20:25:30.844251+00:00,2026-04-07T20:14:07.657+00:00,MX1335-bajaj-avenger-220-cruise,Bajaj,Avenger 220 Cruise,Avenger 220 Cruise,2025.0,2025,available,"51,499.00",51499.0,54499.0,3000.0,51499.0,0.0,oferta,MXN,same,visible,https://www.motosbajaj.com.mx/product-page/avenger-220,https://www.galgo.com/mx/motos/MX1335-bajaj-avenger-220-cruise
1,NaN,NaN,MX250-bajaj-boxer-bm-150,Bajaj,Boxer BM 150,NaN,2024.0,<NA>,no_stock,NaN,NaN,30999.0,0.0,30999.0,NaN,NaN,NaN,NaN,NaN,NaN,https://www.galgo.com/mx/motos/MX250-bajaj-boxer-bm-150
2,NaN,NaN,MX250-bajaj-boxer-bm-150,Bajaj,Boxer BM 150,NaN,2025.0,<NA>,no_stock,NaN,NaN,34499.0,3000.0,31499.0,NaN,NaN,NaN,NaN,NaN,NaN,https://www.galgo.com/mx/motos/MX250-bajaj-boxer-bm-150
3,2026-04-07T20:25:30.844251+00:00,2026-04-07T20:13:45.405+00:00,NaN,NaN,NaN,Boxer BM 150,NaN,2026,NaN,"31,499.00",31499.0,NaN,NaN,NaN,NaN,oferta,MXN,same,visible,https://www.motosbajaj.com.mx/product-page/boxer-bm-150,https://www.galgo.com/mx/motos/nan
4,2026-04-07T20:25:30.844251+00:00,2026-04-07T20:14:14.43+00:00,MX246-bajaj-dominar-250-ug,Bajaj,Dominar 250,Dominar 250 2026,2026.0,2026,available,"73,499.00",73499.0,73499.0,0.0,73499.0,0.0,oferta,MXN,same,visible,https://www.motosbajaj.com.mx/product-page/dominar-250,https://www.galgo.com/mx/motos/MX246-bajaj-dominar-250-ug
5,NaN,NaN,MX3035-bajaj-dominar-400-4e,Bajaj,Dominar 400 4E,NaN,2025.0,<NA>,available,NaN,NaN,97999.0,0.0,97999.0,NaN,NaN,NaN,NaN,NaN,NaN,https://www.galgo.com/mx/motos/MX3035-bajaj-dominar-400-4e
6,2026-04-07T20:25:30.844251+00:00,2026-04-07T20:13:45.333+00:00,MX253-bajaj-dominar-400-te,Bajaj,Dominar 400 TE,Dominar 400 Touring,2025.0,2025,no_stock,"92,999.00",92999.0,95999.0,3000.0,92999.0,0.0,oferta,MXN,same,visible,https://www.motosbajaj.com.mx/product-page/dominar-400,https://www.galgo.com/mx/motos/MX253-bajaj-dominar-400-te
7,2026-04-07T20:25:30.844251+00:00,2026-04-07T20:13:52.6+00:00,MX253-bajaj-dominar-400-te,Bajaj,Dominar 400 TE,Dominar 400 Touring 2026,2026.0,2026,no_stock,"95,999.00",95999.0,95999.0,0.0,95999.0,0.0,oferta,MXN,same,visible,https://www.motosbajaj.com.mx/product-page/dominar-400-touring-2026,https://www.galgo.com/mx/motos/MX253-bajaj-dominar-400-te
8,NaN,NaN,MX2784-bajaj-maxima-z-fl-pasajeros,Bajaj,Maxima Z FL Pasajeros,NaN,2025.0,<NA>,no_stock,NaN,NaN,114000.0,0.0,114000.0,NaN,NaN,NaN,NaN,NaN,NaN,https://www.galgo.com/mx/motos/MX2784-bajaj-maxima-z-fl-pasajeros
9,2026-04-07T20:25:30.844251+00:00,2026-04-07T20:13:59.777+00:00,NaN,NaN,NaN,Maxima Z FL Pasajeros,NaN,2026,NaN,"116,500.00",116500.0,NaN,NaN,NaN,NaN,oferta,MXN,same,visible,https://www.motosbajaj.com.mx/product-page/maxima-z-fl-pasajeros,https://www.galgo.com/mx/motos/nan


In [21]:
df_final[df_final["model"] == "Boxer BM 150"]

,captured_at,previous_scrape_at,code,brand,model,model_name,year,year_scraped,status,price_scraped,price_scraped_with_galgo_fee,price_base,discount_amount,price_net,price_diff,price_type,currency,change_status,visibility,url_scraped,marketplace_url
1,NaN,NaN,MX250-bajaj-boxer-bm-150,Bajaj,Boxer BM 150,NaN,2024.0,<NA>,no_stock,NaN,NaN,30999.0,0.0,30999.0,NaN,NaN,NaN,NaN,NaN,NaN,https://www.galgo.com/mx/motos/MX250-bajaj-boxer-bm-150
2,NaN,NaN,MX250-bajaj-boxer-bm-150,Bajaj,Boxer BM 150,NaN,2025.0,<NA>,no_stock,NaN,NaN,34499.0,3000.0,31499.0,NaN,NaN,NaN,NaN,NaN,NaN,https://www.galgo.com/mx/motos/MX250-bajaj-boxer-bm-150


## 9. Inspeccion

In [22]:
# Diagnostico rapido de cobertura del merge
print(df_merged["_merge"].value_counts(dropna=False))

solo_marketplace = df_merged[df_merged["_merge"] == "left_only"][["brand", "model", "year"]].drop_duplicates()
solo_scrapeado = df_merged[df_merged["_merge"] == "right_only"][["model_name", "model_mapped", "year_scraped", "url"]].drop_duplicates()

print(f"Solo marketplace: {len(solo_marketplace)}")
print(f"Solo scrapeado: {len(solo_scrapeado)}")
solo_scrapeado.head(20)

_merge
both          18
left_only      9
right_only     5
Name: count, dtype: int64
Solo marketplace: 9
Solo scrapeado: 5


,model_name,model_mapped,year_scraped,url
3,Boxer BM 150,boxer bm 150,2026,https://www.motosbajaj.com.mx/product-page/boxer-bm-150
9,Maxima Z FL Pasajeros,maxima z fl pasajeros,2026,https://www.motosbajaj.com.mx/product-page/maxima-z-fl-pasajeros
11,Pulsar LS 125,pulsar ls 125,2025,https://www.motosbajaj.com.mx/product-page/pulsar-ls-125
18,Pulsar N 250 FI + ABS UG 2026,pulsar n 250 fi abs ug,2026,https://www.motosbajaj.com.mx/product-page/pulsar-n-250-fi-abs-ug-2026
31,RE4S Pasajeros,re4s pasajeros,2026,https://www.motosbajaj.com.mx/product-page/re4s-pasajeros


In [23]:
# Filas sin codigo de inventario
df_final[df_final["code"].isna()]

,captured_at,previous_scrape_at,code,brand,model,model_name,year,year_scraped,status,price_scraped,price_scraped_with_galgo_fee,price_base,discount_amount,price_net,price_diff,price_type,currency,change_status,visibility,url_scraped,marketplace_url
3,2026-04-07T20:25:30.844251+00:00,2026-04-07T20:13:45.405+00:00,NaN,NaN,NaN,Boxer BM 150,NaN,2026,NaN,"31,499.00",31499.0,NaN,NaN,NaN,NaN,oferta,MXN,same,visible,https://www.motosbajaj.com.mx/product-page/boxer-bm-150,https://www.galgo.com/mx/motos/nan
9,2026-04-07T20:25:30.844251+00:00,2026-04-07T20:13:59.777+00:00,NaN,NaN,NaN,Maxima Z FL Pasajeros,NaN,2026,NaN,"116,500.00",116500.0,NaN,NaN,NaN,NaN,oferta,MXN,same,visible,https://www.motosbajaj.com.mx/product-page/maxima-z-fl-pasajeros,https://www.galgo.com/mx/motos/nan
11,2026-04-07T20:25:30.844251+00:00,2026-04-07T20:13:45.451+00:00,NaN,NaN,NaN,Pulsar LS 125,NaN,2025,NaN,"29,999.00",29999.0,NaN,NaN,NaN,NaN,oferta,MXN,same,visible,https://www.motosbajaj.com.mx/product-page/pulsar-ls-125,https://www.galgo.com/mx/motos/nan
18,2026-04-07T20:25:30.844251+00:00,2026-04-07T20:14:07.486+00:00,NaN,NaN,NaN,Pulsar N 250 FI + ABS UG 2026,NaN,2026,NaN,"66,499.00",66499.0,NaN,NaN,NaN,NaN,oferta,MXN,same,visible,https://www.motosbajaj.com.mx/product-page/pulsar-n-250-fi-abs-ug-2026,https://www.galgo.com/mx/motos/nan
31,2026-04-07T20:25:30.844251+00:00,2026-04-07T20:14:00.04+00:00,NaN,NaN,NaN,RE4S Pasajeros,NaN,2026,NaN,"99,000.00",99000.0,NaN,NaN,NaN,NaN,oferta,MXN,same,visible,https://www.motosbajaj.com.mx/product-page/re4s-pasajeros,https://www.galgo.com/mx/motos/nan


In [24]:
from datetime import date

today_str = date.today().strftime("%d%m%Y")
print(today_str)

07042026


## 10. Exportar

In [25]:
brand_slug = re.sub(r"[^a-z0-9]+", "_", BRAND_NAME.lower()).strip("_")
output_file = f"{today_str}-{brand_slug}_precios.csv"
df_final.sort_values(by="model_name").to_csv(output_file, index=False)
print(f"Archivo exportado: {output_file} — {len(df_final)} filas")

Archivo exportado: 07042026-bajaj_precios.csv — 32 filas


---
## Anexo: correr sobre catalogo completo

In [ ]:
# # Descomentar para correr con una lista mas grande de URLs
# rows_full = run_price_tracking(
#     urls=urls_to_scrape,
#     brand_name=BRAND_NAME,
# )
# df_full = pd.DataFrame(rows_full)
# print(f"Modelos procesados: {len(df_full)}")
# print(df_full["change_status"].value_counts().to_dict())
# df_full.head()